# Factor research notebook

Interactive companion to `main.py report`: inspect the data, the factor scores on any date, and whether each factor actually ranked stocks correctly (information coefficient).

Run from the project root: `jupyter notebook notebooks/01_factor_research.ipynb` (`pip install notebook` first).

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import pandas as pd, matplotlib.pyplot as plt
from config.settings import load_config
from backtesting.pipeline import load_context, run_backtest, summarize
from reporting.attribution import factor_ic

cfg = load_config()
ctx = load_context(cfg)          # uses cached data in data/cache/
ctx.md.adj_close.shape

## 1. Data quality
Large daily moves are flagged, not deleted (they're often real: earnings, spin-offs).

In [ ]:
ctx.md.quality_flags.sort_values('return').head(10)

In [ ]:
ctx.fundamentals.coverage.mean().round(2)  # share of days with point-in-time fundamental data

## 2. Factor scores on a single date
Everything here uses only data available at that day's close.

In [ ]:
from strategy.factors import FactorModel
model = FactorModel(ctx.features(cfg.strategy), cfg.strategy)
date = ctx.md.dates[-1]
scores = model.score(date)
scores[['sector','score','z_momentum','z_trend','z_sector','z_quality','z_value','z_growth','z_risk']].head(15).round(2)

## 3. Did the factors work? Monthly rank IC

In [ ]:
res = run_backtest(cfg, ctx)
ic = factor_ic(res.scores, ctx.md.adj_close)
ic.rolling(12).mean()[['z_momentum','z_quality','z_value','z_growth','score']].plot(figsize=(10,4), title='12-month rolling mean IC')
plt.axhline(0, color='k', lw=0.8);

In [ ]:
(ic.mean() / ic.std() * len(ic) ** 0.5).round(2).rename('IC t-stat')

## 4. Strategy vs. SPY

In [ ]:
s = summarize(res, ctx, cfg)
pd.DataFrame(s).loc[['cagr','volatility','sharpe','max_drawdown','beta','alpha']]